# Data Quality Audit — 03 · Saudi Tourism Dataset 2015–2024 (Kaggle)

**Source:** `data/raw/tourism_statistics/tourism_data.csv`

**Purpose in the concierge:** Tourism demand / spending / overnight stays by province, year, type.

Standardized audit covering:

```
Dataset
├── Shape
├── Columns & data types
├── Missing values
├── Duplicates
├── Invalid values
├── Outliers
├── Inconsistent categories
├── Geographic validity
├── Date/time validity
├── Data-source/license
└── Known limitations
```

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)

# Resolve repo root whether run from the audit folder or the repo root.
p = Path.cwd()
while p != p.parent and not (p / "data" / "raw").exists():
    p = p.parent
ROOT = p
print("repo root:", ROOT)

# Saudi Arabia bounding box (approx) for geographic validity checks.
SA_LAT = (16.0, 32.5)
SA_LON = (34.5, 56.0)

def iqr_outliers(series):
    """Return (count, lower, upper) of IQR outliers in a numeric series."""
    s = pd.to_numeric(series, errors="coerce").dropna()
    if s.empty:
        return 0, np.nan, np.nan
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return int(((s < lo) | (s > hi)).sum()), round(lo, 2), round(hi, 2)

def missing_report(df):
    m = pd.DataFrame({"missing": df.isna().sum(),
                      "missing_%": (df.isna().mean() * 100).round(1)})
    return m[m["missing"] > 0].sort_values("missing", ascending=False)

def dtype_report(df):
    return pd.DataFrame({
        "dtype": [str(t) for t in df.dtypes],
        "non_null": df.notna().sum().values,
        "n_unique": [df[c].nunique(dropna=True) for c in df.columns],
    }, index=df.columns)


repo root: /home/user/saudi-Digital-Concierge


In [2]:
df = pd.read_csv(ROOT / "data/raw/tourism_statistics/tourism_data.csv")
NUM = ["Tourists_Number","Overnight_Stay","Tourists_Spending",
       "Avg_Stay","Avg_Spending_Trip","Avg_Spending_Night"]
PROVINCES = ["Albaha","Alqassim","Aseer","Eastern_region","Hail","Jazan",
             "Jouf","Madinah","Makkah","Najran","Northern_borders","Riyadh","Tabuk"]
print("Loaded tourism stats:", df.shape)
df.head(3)

Loaded tourism stats: (1058, 9)


,YEARS,Tourists_Number,Overnight_Stay,Tourists_Spending,Avg_Stay,Avg_Spending_Trip,Avg_Spending_Night,Province,Tourism_Type
0,2015,6.0,59.0,43.0,10.0,7029.0,727.0,Albaha,Inbound
1,2015,6.0,59.0,43.0,10.0,7029.0,727.0,Albaha,Inbound
2,2015,0.0,0.0,0.0,0.0,0.0,0.0,Albaha,Inbound


## Shape

In [3]:
print("Rows:", len(df), "| Columns:", df.shape[1])

Rows: 1058 | Columns: 9


## Columns & data types

In [4]:
dtype_report(df)

,dtype,non_null,n_unique
YEARS,int64,1058,10
Tourists_Number,float64,1058,692
Overnight_Stay,float64,1058,865
Tourists_Spending,float64,1058,726
Avg_Stay,float64,1058,220
Avg_Spending_Trip,float64,1058,870
Avg_Spending_Night,float64,1058,517
Province,str,1058,13
Tourism_Type,str,1058,2


## Missing values

In [5]:
missing_report(df) if not missing_report(df).empty else print('No NaN values')

No NaN values


## Duplicates
Exact duplicates exist; and the (year, province, type) key is **not unique** (2–6 rows per group).

In [6]:
print("Exact duplicate rows:", df.duplicated().sum())
g = df.groupby(["YEARS","Province","Tourism_Type"]).size()
print("Rows per (year,province,type) — distribution:")
print(g.value_counts().sort_index().to_string())

Exact duplicate rows: 28
Rows per (year,province,type) — distribution:
2     9
3    89
4    64
5    65
6    32


## Invalid values
Metrics must be ≥ 0; zero rows likely encode suppressed/missing data.

In [7]:
print("Negative values:", int((df[NUM] < 0).sum().sum()))
print("All-zero metric rows:", int((df[NUM].sum(axis=1) == 0).sum()))
print("Rows with Tourists_Number == 0:", int((df["Tourists_Number"] == 0).sum()))

Negative values: 0
All-zero metric rows: 20
Rows with Tourists_Number == 0: 21


## Outliers

In [8]:
for col in NUM:
    n, lo, hi = iqr_outliers(df[col])
    print(f"{col:20s} IQR outliers={n:4d} (bounds {lo}..{hi})")

Tourists_Number      IQR outliers= 139 (bounds -1825.07..3119.12)
Overnight_Stay       IQR outliers= 129 (bounds -12878.12..21960.88)
Tourists_Spending    IQR outliers= 150 (bounds -2650.12..4584.88)
Avg_Stay             IQR outliers= 117 (bounds -2.3..15.3)
Avg_Spending_Trip    IQR outliers=  76 (bounds -1906.0..6044.0)
Avg_Spending_Night   IQR outliers=  75 (bounds -236.5..847.5)


## Inconsistent categories
Province and Tourism_Type controlled vocabularies.

In [9]:
print("Provinces (%d):" % df["Province"].nunique())
print(sorted(df["Province"].unique()))
print("Unexpected provinces:", set(df["Province"].unique()) - set(PROVINCES))
print("\nTourism_Type:", df["Tourism_Type"].value_counts().to_dict())

Provinces (13):
['Albaha', 'Alqassim', 'Aseer', 'Eastern_region', 'Hail', 'Jazan', 'Jouf', 'Madinah', 'Makkah', 'Najran', 'Northern_borders', 'Riyadh', 'Tabuk']
Unexpected provinces: set()

Tourism_Type: {'Domestic': 540, 'Inbound': 518}


## Geographic validity
No coordinates; geography is the `Province` field. Note names differ from other sources (`Eastern_region`, `Northern_borders`).

In [10]:
print("Province is the geographic key; needs mapping to a canonical province name.")

Province is the geographic key; needs mapping to a canonical province name.


## Date/time validity
`YEARS` should be 2015–2024 integers.

In [11]:
print("Year range:", df["YEARS"].min(), "-", df["YEARS"].max())
print("Non-integer / out-of-range years:", int((~df["YEARS"].between(2015, 2024)).sum()))
print(df["YEARS"].value_counts().sort_index().to_string())

Year range: 2015 - 2024
Non-integer / out-of-range years: 0
YEARS
2015    106
2016    106
2017    101
2018    106
2019    107
2020    105
2021    106
2022    106
2023    107
2024    108


## Data-source / license
- **Source:** Kaggle — *Saudi Tourism Dataset 2015–2024*.
- **License:** TBD.
- **Currency:** static snapshot (2015–2024).

## Known limitations
- **Non-unique granularity**: 2–6 rows per (year, province, type) — a sub-category dimension seems flattened. Resolve before aggregating.
- **~28 exact duplicates** and **~20 all-zero rows** (treat zeros as missing).
- Province names differ from other sources → canonical mapping needed.
- Units (thousands / millions SAR) to confirm from the source page.